# Faustine & Pereira (SmartGridComm 2024) — SPS-UK 单 Notebook 复现

论文：**Conformal Multilayer Perceptron-Based Probabilistic Net-Load Forecasting for Low-Voltage Distribution Systems with Photovoltaic Generation**

这份 Notebook 的目标不是“封装一个可复用代码库”，而是把**论文理解、数据下载、预处理、模型、Split Conformal、两类 baseline、指标、图和复现假设**尽量放在一个地方，方便逐格阅读和修改。

核心路线：

**SPS-UK 原始数据 → Net Load → MLPF 点预测 → Split Conformal → 预测区间 → PICP/NMPI**

并复现论文的两组比较：

1. absolute residual vs signed residual non-conformity score；
2. Conformal-MLPF vs MLP-QR vs MLP-MCD。

> 说明：本文只有 6 页，一些实现细节引用作者前作而未在本文完整给出。Notebook 中把“论文明确给出的设置”和“复现假设”分开标注，不把猜测包装成原论文设置。

## 0. 论文设置与复现假设

| 项目 | 论文明确给出 | 本 Notebook |
|---|---|---|
| 数据 | SPS-UK，Plymouth，30 min | 使用公开 set4 数据 |
| 预测对象 | integrated net-load | NetLoad = Load - PV |
| 输入 | 历史 net-load + weather + time covariates | 对齐实现 |
| 结构 | past encoder + future encoder + MLP | clean-room 双编码器近似 |
| Hidden layers | 2 × 256 | 保留 |
| Activation | SiLU | 保留 |
| Optimizer | Adam, lr=0.001 | 保留 |
| LR schedule | 75%/90% 处 ×0.1 | 保留 |
| Point loss | L1/L2 mixed loss | λ 未给出，默认 0.5 |
| Conformal | Split CP，horizon-wise | 保留 |
| Score | absolute / signed residual | absolute 按公式；signed 两侧区间为显式复现假设 |
| Backtest | 10-fold expanding window；初始 ≥12 months；future window 6 months；step 3 months | 保留 |
| Train/Cal | pre-test 数据中 90% train / 10% calibration | 保留 |
| Horizon | multi-step | 取 48×30min=24h，作为复现设定 |
| Lookback | 本文未明确最终值 | 默认 96×30min=48h，作为复现设定 |
| CWE | 论文给了组合公式，但 γ 变换细节不足 | **不伪造精确 CWE**；主要比较 NRMSE/PICP/NMPI |
| RoPE | 图中出现，但本文不足以还原准确张量语义 | 不实现，明确记录偏差 |
| MERRA-2 | 论文称使用 MERRA-2 weather | 公开 set4 weather 为 hourly reanalysis；不能确认与作者提取完全一致 |

这是一份**方法级复现（methodological reproduction）**。如果最后数字与 Table I 不完全一致，优先检查天气数据、作者前作 MLPF 细节、lookback、训练轮数和随机性，而不是先怀疑 SCP 公式。

In [ ]:
# 如缺依赖，在 Jupyter 里取消下一行注释运行
# %pip install -q numpy pandas matplotlib scikit-learn requests torch tqdm

from pathlib import Path
import hashlib
import math
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

SEED = 42
ROOT = Path.cwd()
if ROOT.name != "2024-smartgridcomm-conformal-mlpf":
    # 从仓库根目录启动 notebook 时也能定位
    candidate = ROOT / "papers" / "2024-smartgridcomm-conformal-mlpf"
    if candidate.exists():
        ROOT = candidate

DATA_DIR = ROOT / "data" / "raw" / "sps-uk"
DATA_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("ROOT =", ROOT)
print("DEVICE =", DEVICE)

## 1. 下载 SPS-UK

公开数据来源：Zenodo Record **5500457**，标题 *Electricity demand data and solar generation data from Plymouth. UK*。

论文用到的训练期数据：
- `demand_train_set4.csv`
- `pv_train_set4.csv`
- `weather_train_set4.csv`

为了提高可运行性，代码先尝试 Zenodo；失败时退回到 WPD challenge 的公开 GitHub 镜像。数据不提交到本仓库。

In [ ]:
FILE_SPECS = {
    "demand_train_set4.csv": {
        "md5": "5634ebaaf32b1f130c2cfef1633bb5b1",
        "zenodo": "https://zenodo.org/records/5500457/files/demand_train_set4.csv?download=1",
        "mirror": "https://raw.githubusercontent.com/AyrtonB/WPD-DS-Challenge/main/data/raw/demand_train_set4.csv",
    },
    "pv_train_set4.csv": {
        "md5": "b4d27026390e80c80fdb650a231f4af2",
        "zenodo": "https://zenodo.org/records/5500457/files/pv_train_set4.csv?download=1",
        "mirror": "https://raw.githubusercontent.com/AyrtonB/WPD-DS-Challenge/main/data/raw/pv_train_set4.csv",
    },
    "weather_train_set4.csv": {
        "md5": "fafec5052819225073d185a153289eee",
        "zenodo": "https://zenodo.org/records/5500457/files/weather_train_set4.csv?download=1",
        "mirror": "https://raw.githubusercontent.com/AyrtonB/WPD-DS-Challenge/main/data/raw/weather_train_set4.csv",
    },
}

def md5sum(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def download_one(name, spec):
    dst = DATA_DIR / name
    if dst.exists() and dst.stat().st_size > 0:
        if md5sum(dst) == spec["md5"]:
            print("exists + md5 ok:", name)
            return dst
        print("existing file md5 mismatch; re-download:", name)
        dst.unlink()

    last_error = None
    for url in [spec["zenodo"], spec["mirror"]]:
        try:
            print("downloading:", url)
            with requests.get(url, stream=True, timeout=120) as r:
                r.raise_for_status()
                with open(dst, "wb") as f:
                    for chunk in r.iter_content(1024 * 1024):
                        if chunk:
                            f.write(chunk)
            if md5sum(dst) != spec["md5"]:
                raise RuntimeError(f"MD5 mismatch for {name}")
            print("saved:", dst, f"({dst.stat().st_size/1024/1024:.2f} MB)")
            return dst
        except Exception as e:
            last_error = e
            if dst.exists():
                dst.unlink()
            print("  failed:", repr(e))
    raise RuntimeError(f"Could not download {name}") from last_error

for name, spec in FILE_SPECS.items():
    download_one(name, spec)

## 2. 读取并对齐数据

WPD challenge 原始列名在公开代码中可核对：

- demand: `demand_MW`
- PV: `pv_power_mw`, `irradiance_Wm-2`, `panel_temp_C`
- weather: 6 个 `temp_location*` 与 6 个 `solar_location*`

论文采用 **integrated** 路线，不分别预测 Load 和 PV，而是先构造：

[
NetLoad_t = Load_t - PV_t
]

然后直接预测未来 net-load。

In [ ]:
def read_indexed_csv(path):
    df = pd.read_csv(path)
    if "datetime" not in df.columns:
        raise ValueError(f"{path.name}: expected a datetime column, got {list(df.columns)}")
    df["datetime"] = pd.to_datetime(df["datetime"], utc=True)
    return df.set_index("datetime").sort_index()

demand_raw = read_indexed_csv(DATA_DIR / "demand_train_set4.csv")
pv_raw = read_indexed_csv(DATA_DIR / "pv_train_set4.csv")
weather_raw = read_indexed_csv(DATA_DIR / "weather_train_set4.csv")

print("demand:", demand_raw.shape, demand_raw.columns.tolist())
print("pv:", pv_raw.shape, pv_raw.columns.tolist())
print("weather:", weather_raw.shape, weather_raw.columns.tolist())
display(demand_raw.head(3))
display(pv_raw.head(3))
display(weather_raw.head(3))

In [ ]:
# 论文说 weather 为小时级并插值到 30 min。
# 这里对 weather 先建完整 30-min 索引，再做时间插值。
full_weather_idx = pd.date_range(
    weather_raw.index.min(), weather_raw.index.max(), freq="30min", tz="UTC"
)
weather_30 = weather_raw.reindex(full_weather_idx).interpolate(method="time").ffill().bfill()

temp_cols = [c for c in weather_30.columns if c.startswith("temp_location")]
solar_cols = [c for c in weather_30.columns if c.startswith("solar_location")]

weather_30["temperature"] = weather_30[temp_cols].mean(axis=1)
weather_30["irradiance"] = weather_30[solar_cols].mean(axis=1)

# demand/PV 已是半小时粒度；只取模型真正需要的列
if "demand_MW" not in demand_raw.columns:
    raise KeyError(f"demand_MW not found: {demand_raw.columns.tolist()}")
required_pv = {"pv_power_mw", "irradiance_Wm-2", "panel_temp_C"}
missing = required_pv - set(pv_raw.columns)
if missing:
    raise KeyError(f"PV columns missing: {missing}")

df = pd.concat([
    demand_raw[["demand_MW"]],
    pv_raw[["pv_power_mw", "irradiance_Wm-2", "panel_temp_C"]],
    weather_30[["temperature", "irradiance"]],
], axis=1)

# 小缺口优先做短程时间插值；长缺口直接丢弃，避免大量人工填充影响复现解释
df = df.sort_index()
df = df.interpolate(method="time", limit=2).dropna()

df["net_load_mw"] = df["demand_MW"] - df["pv_power_mw"]

print(df.shape)
print(df.index.min(), "->", df.index.max())
display(df.head())

### 与论文原实现可能存在的预处理差异

作者相关公开代码对 WPD 数据还做过：
- 缺失值修补；
- PV 异常点检测；
- 个别日期人工替换；
- Random Forest 插补。

SmartGridComm 2024 本文并没有把这些细节全部重新写出来。

为了让本 Notebook 的因果链清楚，这里采用**最小透明预处理**：时间对齐、短缺口插值、其余缺失删除。若目标从“方法复现”升级为“Table I 数字级复现”，应再逐项对齐作者前作的数据清洗。

In [ ]:
def add_time_features(x):
    out = x.copy()
    idx = out.index
    hour = idx.hour + idx.minute / 60
    dow = idx.dayofweek
    dom = idx.day - 1

    # 论文说 date-time features 使用 sin/cos 变换
    out["hour_sin"] = np.sin(2*np.pi*hour/24)
    out["hour_cos"] = np.cos(2*np.pi*hour/24)
    out["dow_sin"] = np.sin(2*np.pi*dow/7)
    out["dow_cos"] = np.cos(2*np.pi*dow/7)
    out["dom_sin"] = np.sin(2*np.pi*dom/31)
    out["dom_cos"] = np.cos(2*np.pi*dom/31)
    out["session"] = ((hour >= 6) & (hour < 18)).astype(float)
    return out

df = add_time_features(df)

PAST_FEATURES = [
    "net_load_mw", "irradiance", "temperature",
    "hour_sin", "hour_cos", "dow_sin", "dow_cos",
    "dom_sin", "dom_cos", "session"
]
FUTURE_FEATURES = [
    "irradiance", "temperature",
    "hour_sin", "hour_cos", "dow_sin", "dow_cos",
    "dom_sin", "dom_cos", "session"
]

display(df[["demand_MW","pv_power_mw","net_load_mw","irradiance","temperature"]].describe().T)

In [ ]:
# 对应论文 Fig. 3 的直觉：按一天中的时间画平均净负荷曲线，并看年份差异
plot_df = df[["net_load_mw"]].copy()
plot_df["year"] = plot_df.index.year
plot_df["half_hour"] = plot_df.index.hour + plot_df.index.minute / 60

plt.figure(figsize=(9,4))
for year, g in plot_df.groupby("year"):
    daily = g.groupby("half_hour")["net_load_mw"].mean()
    plt.plot(daily.index, daily.values, label=str(year))
plt.xlabel("Hour of day")
plt.ylabel("Net load (MW)")
plt.title("SPS-UK mean daily net-load profile by year")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## 3. 构造多步预测样本

设：
- lookback (L=96)：过去 48 小时（复现假设）；
- horizon (H=48)：未来 24 小时；
- past encoder 看历史 net-load + 历史协变量；
- future encoder 看未来已知/可获得的 covariates。

注意：论文实际运行时未来 weather 的来源可理解为天气预报/再分析对齐；这里为了复现预测器结构，直接使用数据集里的未来 weather covariates。这一点在研究使用时必须与真实可用信息边界保持一致。

In [ ]:
LOOKBACK = 96
HORIZON = 48

class WindowDataset(Dataset):
    def __init__(self, past, future, target):
        self.past = torch.tensor(past, dtype=torch.float32)
        self.future = torch.tensor(future, dtype=torch.float32)
        self.target = torch.tensor(target, dtype=torch.float32)
    def __len__(self):
        return len(self.target)
    def __getitem__(self, i):
        return self.past[i], self.future[i], self.target[i]

def build_windows(frame):
    p = frame[PAST_FEATURES].to_numpy(np.float32)
    f = frame[FUTURE_FEATURES].to_numpy(np.float32)
    y = frame["net_load_mw"].to_numpy(np.float32)
    ts = frame.index.to_numpy()

    past, future, target, start_ts, end_ts = [], [], [], [], []
    for i in range(LOOKBACK, len(frame)-HORIZON+1):
        past.append(p[i-LOOKBACK:i])
        future.append(f[i:i+HORIZON])
        target.append(y[i:i+HORIZON])
        start_ts.append(ts[i])
        end_ts.append(ts[i+HORIZON-1])
    return {
        "past": np.asarray(past),
        "future": np.asarray(future),
        "target": np.asarray(target),
        "start_ts": pd.to_datetime(np.asarray(start_ts)),
        "end_ts": pd.to_datetime(np.asarray(end_ts)),
    }

arr = build_windows(df)
print({k: v.shape if hasattr(v, "shape") else len(v) for k,v in arr.items()})

## 4. MLPF 的 clean-room 近似

论文的 MLPF 不是通用术语，而是作者前作中的 **MLP-based Forecast** 框架。

本文图中给出的关键结构是：
- past encoder (g_phi)
- future encoder (h_phi)
- 两个 representation 融合
- MLP 预测未来 H 步

本文图里还出现 RoPE / LayerNorm，但 6 页论文不足以精确还原 RoPE 的张量语义。因此这里保留双编码器与 LayerNorm，不自行“脑补” RoPE。

In [ ]:
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

class MLPEncoder(nn.Module):
    def __init__(self, in_dim, hidden=256, layers=2, dropout=0.0):
        super().__init__()
        blocks = [nn.LayerNorm(in_dim)]
        d = in_dim
        for _ in range(layers):
            blocks += [
                nn.Linear(d, hidden),
                nn.BatchNorm1d(hidden),
                nn.SiLU(),
                nn.Dropout(dropout),
            ]
            d = hidden
        self.net = nn.Sequential(*blocks)
    def forward(self, x):
        return self.net(x)

class PointMLPF(nn.Module):
    def __init__(self, point_dropout=0.0):
        super().__init__()
        self.past = MLPEncoder(LOOKBACK*len(PAST_FEATURES), 256, 2, point_dropout)
        self.future = MLPEncoder(HORIZON*len(FUTURE_FEATURES), 256, 2, point_dropout)
        self.fusion = nn.Sequential(nn.LayerNorm(256), nn.SiLU())
        self.head = nn.Linear(256, HORIZON)
    def encode(self, past, future):
        z = self.past(past.flatten(1)) + self.future(future.flatten(1))
        return self.fusion(z)
    def forward(self, past, future):
        return self.head(self.encode(past, future))

class QuantileMLPF(PointMLPF):
    def __init__(self, quantiles=(0.05,0.5,0.95)):
        super().__init__(0.0)
        self.quantiles = tuple(quantiles)
        self.head = nn.Linear(256, HORIZON*len(self.quantiles))
    def forward(self, past, future):
        out = self.head(self.encode(past, future))
        return out.view(-1, HORIZON, len(self.quantiles))

class LastLayerMCDMLPF(PointMLPF):
    # 对齐论文 Fig.2(d) 的“last-layer Monte Carlo dropout”思路
    def __init__(self, p=0.1):
        super().__init__(0.0)
        self.mcd = nn.Dropout(p)
        self.head = nn.Linear(256, HORIZON)
    def forward(self, past, future):
        return self.head(self.mcd(self.encode(past, future)))

def mixed_l1_l2(pred, true, lam=0.5):
    e = true-pred
    return (lam*e.square() + (1-lam)*e.abs()).mean()

def pinball_loss(pred, true, quantiles):
    q = torch.tensor(quantiles, device=pred.device, dtype=pred.dtype).view(1,1,-1)
    e = true.unsqueeze(-1)-pred
    return torch.maximum(q*e, (q-1)*e).mean()

## 5. 时间回测、训练集与校准集

论文写的是：
- 10-fold backtesting cross-validation；
- expanding window；
- 初始历史期至少 12 个月；
- future/test window = 6 个月；
- 每次向前移动 3 个月；
- 每个 fold 的 pre-test 数据中，90% 用于 training，最后 10% 用于 calibration。

这里额外做一个严谨处理：**用 target end timestamp 切分**，确保任何 train/calibration 样本的 48 步目标都不会跨进 test period。

In [ ]:
def make_backtest_folds(arr, n_folds=10):
    starts, ends = arr["start_ts"], arr["end_ts"]
    first = starts.min()
    last = starts.max()
    train_end = first + pd.DateOffset(months=12)

    folds = []
    for fold in range(n_folds):
        test_start = train_end
        test_end = test_start + pd.DateOffset(months=6)

        pre_idx = np.flatnonzero(ends < test_start)
        test_idx = np.flatnonzero((starts >= test_start) & (starts < test_end))
        if len(pre_idx) < 100 or len(test_idx) == 0:
            break

        n_cal = max(1, math.ceil(len(pre_idx)*0.10))
        train_idx = pre_idx[:-n_cal]
        cal_idx = pre_idx[-n_cal:]
        folds.append((fold, train_idx, cal_idx, test_idx, test_start, test_end))
        train_end = train_end + pd.DateOffset(months=3)
    return folds

folds = make_backtest_folds(arr)
for f, tr, ca, te, s, e in folds:
    print(f"fold={f:2d} train={len(tr):5d} cal={len(ca):4d} test={len(te):5d}  {s.date()} -> {e.date()}")

In [ ]:
class FoldScaler:
    def fit(self, past, future, target):
        self.p_mean = past.mean((0,1), keepdims=True)
        self.p_std = past.std((0,1), keepdims=True)
        self.p_std[self.p_std < 1e-8] = 1
        self.f_mean = future.mean((0,1), keepdims=True)
        self.f_std = future.std((0,1), keepdims=True)
        self.f_std[self.f_std < 1e-8] = 1
        self.y_mean = float(target.mean())
        self.y_std = float(target.std()) or 1.0
        return self
    def transform(self, subset):
        return (
            (subset["past"]-self.p_mean)/self.p_std,
            (subset["future"]-self.f_mean)/self.f_std,
            (subset["target"]-self.y_mean)/self.y_std,
        )
    def inv_y(self, y):
        return y*self.y_std+self.y_mean

def subset(idx):
    return {k: arr[k][idx] for k in ["past","future","target"]}

def loader_from(sub, scaler, batch=128, shuffle=False):
    p,f,y = scaler.transform(sub)
    return DataLoader(WindowDataset(p,f,y), batch_size=batch, shuffle=shuffle)

def fit_model(model, train_sub, scaler, epochs=80, lr=1e-3, lam=0.5, quantiles=None):
    model = model.to(DEVICE)
    dl = loader_from(train_sub, scaler, batch=128, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    milestones = sorted({max(1,int(epochs*0.75)), max(1,int(epochs*0.90))})
    sched = torch.optim.lr_scheduler.MultiStepLR(opt, milestones=milestones, gamma=0.1)

    losses = []
    for ep in range(epochs):
        model.train()
        total, n = 0.0, 0
        for p,f,y in dl:
            p,f,y = p.to(DEVICE),f.to(DEVICE),y.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            pred = model(p,f)
            loss = pinball_loss(pred,y,quantiles) if quantiles is not None else mixed_l1_l2(pred,y,lam)
            loss.backward()
            opt.step()
            total += loss.item()*len(y)
            n += len(y)
        sched.step()
        losses.append(total/n)
    return model, losses

@torch.no_grad()
def predict(model, sub, scaler, batch=256):
    dl = loader_from(sub, scaler, batch=batch, shuffle=False)
    model.eval()
    out = []
    for p,f,_ in dl:
        out.append(model(p.to(DEVICE),f.to(DEVICE)).cpu().numpy())
    return scaler.inv_y(np.concatenate(out))

@torch.no_grad()
def predict_mc(model, sub, scaler, draws=100, batch=256):
    dl = loader_from(sub, scaler, batch=batch, shuffle=False)
    # BatchNorm 保持 eval；仅 Dropout 切回 train
    model.eval()
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.train()

    sims = []
    for _ in range(draws):
        one = []
        for p,f,_ in dl:
            one.append(model(p.to(DEVICE),f.to(DEVICE)).cpu().numpy())
        sims.append(scaler.inv_y(np.concatenate(one)))
    return np.asarray(sims)

## 6. Split Conformal：论文最核心、也最简单的部分

absolute residual score：

[
\gamma_{i,h}=|y_{i,h}-\hat y_{i,h}|
]

每个 horizon (h) 单独取有限样本修正后的 (1-\alpha) 分位数：

[
\epsilon_h = Q_{1-\alpha}(\gamma_{1,h},...,\gamma_{n,h})
]

然后：

[
C_h(x)=[\hat y_h-\epsilon_h,\hat y_h+\epsilon_h]
]

注意：**horizon-wise calibration 已经在这篇论文里做了**。后续做 Adaptive CP 时，创新点不能只是“每个 horizon 分开校准”。

In [ ]:
ALPHA = 0.10

def finite_sample_q(scores, alpha=ALPHA):
    scores = np.asarray(scores)
    n = scores.shape[0]
    rank = min(n, max(1, math.ceil((n+1)*(1-alpha))))
    return np.partition(scores, rank-1, axis=0)[rank-1]

def conformal_abs_fit(y_cal, pred_cal, alpha=ALPHA):
    return finite_sample_q(np.abs(y_cal-pred_cal), alpha)

def conformal_abs_interval(pred, q):
    return pred-q, pred+q

def signed_interval_fit(y_cal, pred_cal, alpha=ALPHA):
    # 论文 Eq.(9) 只定义 signed score，没有完整说明两侧 interval transform。
    # 这里用 equal-tail empirical residual quantiles，明确标为复现假设。
    r = y_cal-pred_cal
    lo = np.quantile(r, alpha/2, axis=0, method="lower")
    hi = np.quantile(r, 1-alpha/2, axis=0, method="higher")
    return lo,hi

def signed_interval(pred, lo_q, hi_q):
    return pred+lo_q, pred+hi_q

def nrmse(y,p):
    rmse = np.sqrt(np.mean((y-p)**2))
    rng = np.max(y)-np.min(y)
    return float(rmse/rng)

def picp(y,l,u):
    return float(np.mean((y>=l)&(y<=u)))

def nmpi(y,l,u):
    rng = np.max(y)-np.min(y)
    return float(np.median(u-l)/rng)

def horizon_picp(y,l,u):
    return np.mean((y>=l)&(y<=u), axis=0)

def horizon_width(l,u):
    return np.mean(u-l, axis=0)

## 7. 运行实验

为了 Notebook 易跑，提供两种模式：

- `quick`：只跑 1 个 fold、少量样本、少量 epoch，用来检查流程；
- `full`：按论文思路跑所有可构造 folds，训练更久。

第一次建议 quick。确认数据、GPU、结果都正常后改成 full。

> 这里不会假装“full 就一定等于作者 Table I”。MLPF 内部结构、MERRA-2 提取、若干超参数和清洗细节仍可能不同。

In [ ]:
RUN_MODE = "quick"   # 改成 "full" 做完整复现
EPOCHS = 5 if RUN_MODE=="quick" else 80
MAX_FOLDS = 1 if RUN_MODE=="quick" else len(folds)

results = []
example = None

for fold, train_idx, cal_idx, test_idx, test_start, test_end in folds[:MAX_FOLDS]:
    if RUN_MODE=="quick":
        train_idx = train_idx[-min(len(train_idx), 3000):]
        cal_idx = cal_idx[-min(len(cal_idx), 800):]
        test_idx = test_idx[:min(len(test_idx), 800)]

    train_sub, cal_sub, test_sub = subset(train_idx), subset(cal_idx), subset(test_idx)
    scaler = FoldScaler().fit(train_sub["past"], train_sub["future"], train_sub["target"])

    print(f"\nfold {fold}: train={len(train_idx)}, cal={len(cal_idx)}, test={len(test_idx)}")

    # A. deterministic MLPF
    seed_everything(SEED+fold)
    point, loss_hist = fit_model(PointMLPF(), train_sub, scaler, epochs=EPOCHS)
    cal_pred = predict(point, cal_sub, scaler)
    test_pred = predict(point, test_sub, scaler)
    y_cal, y_test = cal_sub["target"], test_sub["target"]

    # Experiment 1: absolute vs signed
    q = conformal_abs_fit(y_cal, cal_pred)
    abs_l, abs_u = conformal_abs_interval(test_pred, q)
    slo, shi = signed_interval_fit(y_cal, cal_pred)
    sig_l, sig_u = signed_interval(test_pred, slo, shi)

    results.append({"fold":fold,"method":"Conformal-MLPF(abs)",
                    "NRMSE":nrmse(y_test,test_pred),"PICP":picp(y_test,abs_l,abs_u),"NMPI":nmpi(y_test,abs_l,abs_u)})
    results.append({"fold":fold,"method":"Conformal-MLPF(signed)",
                    "NRMSE":nrmse(y_test,test_pred),"PICP":picp(y_test,sig_l,sig_u),"NMPI":nmpi(y_test,sig_l,sig_u)})

    # Experiment 2 baseline: QR
    quantiles = (0.05,0.50,0.95)
    seed_everything(SEED+1000+fold)
    qr, _ = fit_model(QuantileMLPF(quantiles), train_sub, scaler, epochs=EPOCHS, quantiles=quantiles)
    qr_pred = predict(qr, test_sub, scaler)
    qr_l, qr_p, qr_u = qr_pred[...,0], qr_pred[...,1], qr_pred[...,2]
    results.append({"fold":fold,"method":"MLP-QR",
                    "NRMSE":nrmse(y_test,qr_p),"PICP":picp(y_test,qr_l,qr_u),"NMPI":nmpi(y_test,qr_l,qr_u)})

    # Experiment 2 baseline: last-layer MC Dropout
    seed_everything(SEED+2000+fold)
    mcd, _ = fit_model(LastLayerMCDMLPF(p=0.10), train_sub, scaler, epochs=EPOCHS)
    draws = predict_mc(mcd, test_sub, scaler, draws=20 if RUN_MODE=="quick" else 100)
    mcd_p = draws.mean(0)
    mcd_l = np.quantile(draws,0.05,axis=0)
    mcd_u = np.quantile(draws,0.95,axis=0)
    results.append({"fold":fold,"method":"MLP-MCD",
                    "NRMSE":nrmse(y_test,mcd_p),"PICP":picp(y_test,mcd_l,mcd_u),"NMPI":nmpi(y_test,mcd_l,mcd_u)})

    example = dict(
        y=y_test, point=test_pred, abs_l=abs_l, abs_u=abs_u,
        sig_l=sig_l, sig_u=sig_u, qr_l=qr_l, qr_p=qr_p, qr_u=qr_u,
        mcd_l=mcd_l, mcd_p=mcd_p, mcd_u=mcd_u,
    )

metrics = pd.DataFrame(results)
display(metrics)
display(metrics.groupby("method")[["NRMSE","PICP","NMPI"]].mean().round(4))

## 8. 结果图：先看 calibration，而不是只看“谁最好”

概率预测至少同时看：
- **PICP**：区间实际覆盖多少真实值；
- **NMPI**：区间有多宽；
- **NRMSE**：中心/点预测本身误差。

目标 nominal coverage 为 0.90。一个区间如果极宽，PICP 可以很高，但信息价值很低。

In [ ]:
if example is not None:
    # 画一个预测起点的 48 步轨迹
    i = 0
    h = np.arange(1,HORIZON+1)
    plt.figure(figsize=(11,4))
    plt.plot(h, example["y"][i], marker="o", ms=3, label="True")
    plt.plot(h, example["point"][i], label="Point forecast")
    plt.fill_between(h, example["abs_l"][i], example["abs_u"][i], alpha=0.25, label="90% conformal interval")
    plt.xlabel("Forecast horizon (30-min steps)")
    plt.ylabel("Net load (MW)")
    plt.title("Conformal-MLPF: one 24-hour multi-horizon forecast")
    plt.legend()
    plt.grid(alpha=0.2)
    plt.show()

    hp = horizon_picp(example["y"], example["abs_l"], example["abs_u"])
    hw = horizon_width(example["abs_l"], example["abs_u"])

    fig, ax1 = plt.subplots(figsize=(11,4))
    ax1.plot(h, hp, label="Horizon-wise PICP")
    ax1.axhline(0.90, ls="--", label="Nominal 0.90")
    ax1.set_xlabel("Forecast horizon")
    ax1.set_ylabel("PICP")
    ax1.set_ylim(0,1.05)
    ax1.grid(alpha=0.2)
    ax1.legend(loc="upper left")
    ax2 = ax1.twinx()
    ax2.plot(h, hw, alpha=0.6, label="Mean interval width")
    ax2.set_ylabel("Interval width (MW)")
    ax2.legend(loc="upper right")
    plt.title("Coverage and width by forecast horizon")
    plt.show()

## 9. 与论文 Table I 对照

论文对 SPS-UK 报告：

| Model | NRMSE | PICP | NMPI | CWE |
|---|---:|---:|---:|---:|
| MLP-MCD | 0.19 | 0.82 | 0.36 | 0.75 |
| MLP-QR | 0.13 | 0.96 | 0.32 | 0.83 |
| Conformal-MLPF | 0.13 | 0.78 | 0.24 | 0.74 |

这里最值得注意的不是“Conformal 一定最好”，而是：**nominal 0.90 的情况下，论文在 SPS-UK 上报告的 Conformal PICP 只有 0.78。**

这也说明：
- 这篇论文主要证明了“MLP + SCP 在 LV net-load probability forecasting 中可以工作”；
- 并没有证明 static SCP 在所有时间状态下稳定达到 nominal coverage；
- 论文结论中提出 Adaptive CP，更像是基于 static SCP 的理论局限和 net-load 的季节性/异方差性质提出的未来方向，而不是本文已经通过 rolling/seasonal coverage 实验证明的问题。

我们的 Notebook 因此额外保留了 horizon-wise coverage 图，后续很容易继续扩展成：
- rolling coverage；
- monthly / seasonal coverage；
- Split CP vs Adaptive CP。

## 10. 复现边界与下一步

这份 Notebook 已经把论文最核心的“可复现实验对象”集中到一个文件里，但下面这些仍然不能从 SmartGridComm 6 页正文中无歧义恢复：

1. MLPF 前作中的全部结构细节，尤其 RoPE 的精确使用方式；
2. L1/L2 mixing coefficient λ 的最终取值；
3. lookback L 的最终选定值；
4. signed residual 到两侧 prediction interval 的精确实现；
5. MERRA-2 weather 的原始提取与插值版本；
6. CWE 中 γ_PICP / γ_NMPI 的完整变换定义；
7. 作者数据清洗中的所有人工修复与异常点处理。

因此，这个版本适合作为：
- 理解论文；
- 验证 SCP 公式；
- 重跑 SPS-UK；
- 检查 coverage/width；
- 作为后续 Adaptive Conformal 的 static baseline。

如果下一步要做**数字级复现**，最值得优先收紧的是：
**作者原始数据清洗 → MERRA-2 weather → MLPF 前作实现 → lookback/epochs/λ**。

如果下一步是服务我们自己的论文，则不必执着 Table I 完全一致，而应该把这份 Notebook 作为 static Split CP 基线，继续增加 **seasonal / rolling / horizon-wise calibration stability** 与 Adaptive CP。